In [ ]:
/content/drive/MyDrive/AG News Classification Dataset.csv

In [2]:
!pip install -qU gensim


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 66.3 MB/s eta 0:00:00


In [3]:
import numpy as np
import pandas as pd
import re
import pickle
import time

from gensim.models import Word2Vec

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

In [4]:
df = pd.read_csv("/content/drive/MyDrive/AG News Classification Dataset.csv")
df.head()

,Class Index,Title,Description
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli..."
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...
3,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...
4,3,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco..."


In [5]:
df.shape

(120000, 3)

In [6]:
# Class Index -> category name (standard AG News mapping)
label_map = {1: 'World', 2: 'Sports', 3: 'Business', 4: 'Sci/Tech'}
df['category'] = df['Class Index'].map(label_map)

# Combine title + description into a single text field (more context for embeddings)
df['text'] = df['Title'] + ' ' + df['Description']

df = df[['text', 'category']]
df.head()

,text,category
0,Wall St. Bears Claw Back Into the Black (Reute...,Business
1,Carlyle Looks Toward Commercial Aerospace (Reu...,Business
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,Business
3,Iraq Halts Oil Exports from Main Southern Pipe...,Business
4,"Oil prices soar to all-time record, posing new...",Business


In [18]:
# Clean

In [7]:
df['category'].value_counts()

,count
category,
Business,30000
Sci/Tech,30000
Sports,30000
World,30000


In [8]:
df.isnull().sum()

,0
text,0
category,0


In [9]:
df.duplicated().sum()

np.int64(0)

In [10]:
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)
df.shape

(120000, 2)

In [11]:
def clean_text(text):

    text = text.lower()

    text = re.sub(r"http\\S+", " ", text)

    text = re.sub(r"[^a-z ]", " ", text)

    text = re.sub(r"\\s+", " ", text).strip()

    return text.split()

In [12]:
## 3. Train Word2Vec on the corpus

In [13]:
tokenized_sentences = df['text'].apply(clean_text).tolist()

t0 = time.time()
w2v = Word2Vec(
    sentences=tokenized_sentences,
    vector_size=200,
    window=5,
    min_count=2,
    workers=1,
    sg=0,          # CBOW
    epochs=5,
    seed=42
)
print("Trained in", round(time.time() - t0, 2), "seconds")
print("Vocabulary size:", len(w2v.wv.key_to_index))

w2v.save("word2vec_agnews.model")

Trained in 63.05 seconds
Vocabulary size: 42303


In [14]:
# Sanity check: does the model learn sensible neighbours?
print(w2v.wv.most_similar('war'))

[('terrorism', 0.6518685817718506), ('invasion', 0.6294159889221191), ('insurgency', 0.6118133068084717), ('fighting', 0.5775868892669678), ('conflict', 0.5719630122184753), ('fight', 0.5644112229347229), ('violence', 0.5616679191589355), ('genocide', 0.5573582053184509), ('politics', 0.5180475115776062), ('crimes', 0.5149958729743958)]


In [15]:
## 4. Build the embedding lookup dictionary

In [16]:
vocabulary = set()

for sentence in df["text"]:
    vocabulary.update(clean_text(sentence))

embedding_lookup = {}

for word in vocabulary:
    if word in w2v.wv:
        embedding_lookup[word] = w2v.wv[word]

with open("embedding_lookup.pkl", "wb") as f:
    pickle.dump(embedding_lookup, f)

print("Saved Successfully")
print("Vocabulary Size:", len(embedding_lookup))

Saved Successfully
Vocabulary Size: 42303


In [17]:
## 5. Convert each article into a sentence vector (mean of word vectors)

In [19]:
def sentence_vector(sentence):

    words = clean_text(sentence)

    valid_words = [word for word in words if word in w2v.wv]

    if len(valid_words) == 0:

        return np.zeros(200)

    return w2v.wv.get_mean_vector(valid_words)

In [20]:
X = np.array(df['text'].apply(sentence_vector).tolist())
X.shape

(120000, 200)

In [21]:
## 6. Encode labels

In [22]:
encoder = LabelEncoder()

y = encoder.fit_transform(df['category'])

pickle.dump(encoder, open("label_encoder.pkl", "wb"))

list(encoder.classes_)

['Business', 'Sci/Tech', 'Sports', 'World']

In [23]:
## 7. Train / validation / test split + scaling

In [24]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

In [25]:
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.20, random_state=42, stratify=y_train)

In [26]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_val = scaler.transform(X_val)

X_test = scaler.transform(X_test)

pickle.dump(scaler, open("scaler.pkl", "wb"))

X_train.shape, X_val.shape, X_test.shape

((76800, 200), (19200, 200), (24000, 200))

In [28]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 25.5 MB/s eta 0:00:00


In [29]:
import optuna
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Input

from tensorflow.keras.regularizers import l1_l2

from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.optimizers import Nadam

from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ReduceLROnPlateau

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay

import matplotlib.pyplot as plt

In [30]:
def objective(trial):

    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True)

    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "RMSprop", "Nadam"])

    activation = trial.suggest_categorical("activation", ["relu", "tanh", "elu"])

    batch_size = trial.suggest_categorical("batch_size", [64, 128, 256])

    n_layers = trial.suggest_int("n_layers", 2, 4)

    model = Sequential()

    model.add(Input(shape=(200,)))

    for i in range(n_layers):

        units = trial.suggest_categorical(f"units_{i}", [64, 128, 256])

        dropout = trial.suggest_float(f"dropout_{i}", 0.10, 0.50)

        reg = trial.suggest_float(f"reg_{i}", 1e-6, 1e-3, log=True)

        model.add(Dense(
            units=units,
            activation=activation,
            kernel_regularizer=l1_l2(reg, reg)
        ))

        model.add(BatchNormalization())

        model.add(Dropout(dropout))

    model.add(Dense(4, activation="softmax"))

    optimizer_dict = {

        "Adam": Adam(learning_rate),

        "RMSprop": RMSprop(learning_rate),

        "Nadam": Nadam(learning_rate)

    }

    model.compile(

        optimizer=optimizer_dict[optimizer_name],

        loss="sparse_categorical_crossentropy",

        metrics=["accuracy"]

    )

    early_stop = EarlyStopping(

        monitor="val_accuracy",

        patience=3,

        restore_best_weights=True,

        mode="max"

    )

    reduce_lr = ReduceLROnPlateau(

        monitor="val_loss",

        factor=0.5,

        patience=2,

        min_lr=1e-6

    )

    history = model.fit(

        X_train,

        y_train,

        validation_data=(X_val, y_val),

        epochs=12,

        batch_size=batch_size,

        callbacks=[early_stop, reduce_lr],

        verbose=0

    )

    return max(history.history["val_accuracy"])

In [31]:
study = optuna.create_study(

    direction="maximize",

    sampler=optuna.samplers.TPESampler(seed=42)

)

[I 2026-07-03 10:02:01,634] A new study created in memory with name: no-name-ba4e1313-1c41-44fe-95c1-66565e1af5dc


In [32]:
study.optimize(

    objective,

    n_trials=12,

    show_progress_bar=True

)

  0%|          | 0/12 [00:00<?, ?it/s]

[I 2026-07-03 10:02:59,304] Trial 0 finished with value: 0.903333306312561 and parameters: {'learning_rate': 0.0001329291894316216, 'optimizer': 'Adam', 'activation': 'relu', 'batch_size': 64, 'n_layers': 2, 'units_0': 64, 'dropout_0': 0.17272998688284025, 'reg_0': 3.549878832196506e-06, 'units_1': 128, 'dropout_1': 0.21649165607921678, 'reg_1': 6.847920095574779e-05}. Best is trial 0 with value: 0.903333306312561.
[I 2026-07-03 10:03:35,722] Trial 1 finished with value: 0.889843761920929 and parameters: {'learning_rate': 2.621087878265438e-05, 'optimizer': 'Nadam', 'activation': 'relu', 'batch_size': 256, 'n_layers': 2, 'units_0': 256, 'dropout_0': 0.4233589392465845, 'reg_0': 8.200518402245835e-06, 'units_1': 128, 'dropout_1': 0.14881529393791154, 'reg_1': 3.058656666978529e-05}. Best is trial 0 with value: 0.903333306312561.
[I 2026-07-03 10:05:08,710] Trial 2 finished with value: 0.8943750262260437 and parameters: {'learning_rate': 1.2681352169084594e-05, 'optimizer': 'Adam', 'acti

In [33]:
print("Best Validation Accuracy :")

print(study.best_trial.value)

Best Validation Accuracy :
0.903333306312561


In [34]:
best = study.best_trial.params

print(best)

{'learning_rate': 0.0001329291894316216, 'optimizer': 'Adam', 'activation': 'relu', 'batch_size': 64, 'n_layers': 2, 'units_0': 64, 'dropout_0': 0.17272998688284025, 'reg_0': 3.549878832196506e-06, 'units_1': 128, 'dropout_1': 0.21649165607921678, 'reg_1': 6.847920095574779e-05}


In [35]:
## Train the final model on train + validation, evaluate on test

In [36]:
# -----------------------------
# Merge Train + Validation
# -----------------------------

X_full = np.vstack((X_train, X_val))
y_full = np.concatenate((y_train, y_val))

print(X_full.shape)
print(y_full.shape)

# -----------------------------
# Build Model
# -----------------------------

model = Sequential()

model.add(Input(shape=(X_full.shape[1],)))

for i in range(best["n_layers"]):

    model.add(
        Dense(
            units=best[f'units_{i}'],
            activation=best["activation"],
            kernel_regularizer=l1_l2(
                l1=best[f'reg_{i}'],
                l2=best[f'reg_{i}']
            )
        )
    )

    model.add(BatchNormalization())

    model.add(
        Dropout(best[f'dropout_{i}'])
    )

model.add(Dense(4, activation='softmax'))

# -----------------------------
# Compile
# -----------------------------

optimizer_dict = {"Adam": Adam(best["learning_rate"]), "RMSprop": RMSprop(best["learning_rate"]), "Nadam": Nadam(best["learning_rate"])}

model.compile(
    optimizer=optimizer_dict[best["optimizer"]],
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

(96000, 200)
(96000,)


Model: "sequential_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_45 (Dense)                │ (None, 64)             │        12,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_33          │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_33 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_46 (Dense)                │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_34          │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_34 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_47 (Dense)                │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,468 (87.77 KB)

 Trainable params: 22,084 (86.27 KB)

 Non-trainable params: 384 (1.50 KB)

In [37]:
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=6,
    mode='max',
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6
)

history = model.fit(
    X_full,
    y_full,
    validation_split=0.1,
    epochs=60,
    batch_size=best["batch_size"],
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

Epoch 1/60
1350/1350 ━━━━━━━━━━━━━━━━━━━━ 14s 6ms/step - accuracy: 0.8030 - loss: 0.6301 - val_accuracy: 0.8813 - val_loss: 0.4191 - learning_rate: 1.3293e-04
Epoch 2/60
1350/1350 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8655 - loss: 0.4547 - val_accuracy: 0.8898 - val_loss: 0.3844 - learning_rate: 1.3293e-04
Epoch 3/60
1350/1350 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.8750 - loss: 0.4244 - val_accuracy: 0.8942 - val_loss: 0.3688 - learning_rate: 1.3293e-04
Epoch 4/60
1350/1350 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8812 - loss: 0.4041 - val_accuracy: 0.8961 - val_loss: 0.3562 - learning_rate: 1.3293e-04
Epoch 5/60
1350/1350 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8839 - loss: 0.3899 - val_accuracy: 0.8985 - val_loss: 0.3483 - learning_rate: 1.3293e-04
Epoch 6/60
1350/1350 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8875 - loss: 0.3803 - val_accuracy: 0.8999 - val_loss: 0.3417 - learning_rate: 1.3293e-04
Epoch 7/60
1350/1350 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/st

In [38]:
loss, accuracy = model.evaluate(

    X_test,

    y_test,

    verbose=0

)

print(f"\nTest Loss     : {loss:.4f}")
print(f"Test Accuracy : {accuracy:.4f}")


Test Loss     : 0.2849
Test Accuracy : 0.9092


In [39]:
## Predictions and evaluation metrics

In [40]:
model.save("agnews_classification_model.keras")

print("\nModel Saved Successfully!")


Model Saved Successfully!
